In [ ]:
!pip install datasets
!pip install evaluate
!pip install fsspec==2023.9.2

## Imports

In [ ]:
import numpy as np
import nltk
nltk.download('stopwords')

In [ ]:
# Imports
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

## Obtención de datos

Descargamos los datos del [repositorio de Huggingface](https://huggingface.co/datasets/luisgasco/profner_classification_master).

In [ ]:
#NO-MODIFY: DATA LOAD
from datasets import load_dataset, Dataset, DatasetDict, ClassLabel
dataset = load_dataset("luisgasco/profner_classification_master")

El dataset contiene tres subsets:
- **train** y **validation**: Contienen el identificador del tweet, el texto, y su etiqueta, que podrá tener valor 1, si contiene una mención de una profesión; o valor 0, si no contiene una mención de una profesión.
- **test**: El test set tambiíen contiene la información de label por un requerimiento de Huggingface, pero el contenido de esta variable es siempre "-1". Es decir que deberéis predecir nuevas etiquetas una vez hayáis entrenado el modelo utilizando el train y el validation set.

## Análisis exploratorio de datos

Para hacer el análisis exploratorio de datos, transformamos cada subset a un pandas dataframe para mayor comodidad.

In [ ]:
#NO-MODIFY: DATA LOAD
dataset_train_df = dataset["train"].to_pandas()
dataset_val_df = dataset["validation"].to_pandas()
dataset_test_df = dataset["test"].to_pandas()

**Número de documentos**

In [ ]:
def get_num_docs_evaluation(dataset_df):

  num_docs = len(dataset_df)

  return num_docs


In [ ]:
# Aplicar la función

num_train_docs = get_num_docs_evaluation(dataset_train_df)
num_val_docs = get_num_docs_evaluation(dataset_val_df)

print(f"Número de documentos del conjunto de entrenamiento: {num_train_docs}")
print(f"Número de documentos del conjunto de validación: {num_val_docs}")


El conjunto de entrenamiento cuenta con 2.786 documentos y el de validación con 999. Esta distribución permite entrenar el modelo con una cantidad suficiente de datos y utilizar el conjunto de validación para comprobar su capacidad de generalización.

**Número de documentos duplicados**

In [ ]:
def detect_duplicates_evaluation(dataset_df):

  num_duplicates = dataset_df.duplicated().sum()

  return num_duplicates

In [ ]:
# Aplicar la función

num_train_duplicates = detect_duplicates_evaluation(dataset_train_df)
num_val_duplicates = detect_duplicates_evaluation(dataset_val_df)

print(f"Número de documentos duplicados en entrenamiento: {num_train_duplicates}")
print(f"Número de documentos duplicados en validación: {num_val_duplicates}")

No se han encontrado documentos duplicados en ninguno de los dos conjuntos. Esto es positivo, ya que evita que el modelo se entrene con ejemplos repetidos y hace que la evaluación sea más fiable.

**Número de documentos por cada clase:**


In [ ]:
def analyse_num_labels_evaluation(dataset_df):

  num_positives = (dataset_df["label"] == 1).sum()
  num_negatives = (dataset_df["label"] == 0).sum()

  return num_positives, num_negatives

In [ ]:
# Aplicar la función

train_pos, train_neg = analyse_num_labels_evaluation(dataset_train_df)
val_pos, val_neg = analyse_num_labels_evaluation(dataset_val_df)

print("Entrenamiento")
print(f"Clase 1 (positivos): {train_pos}")
print(f"Clase 0 (negativos): {train_neg}")

print("\nValidación")
print(f"Clase 1 (positivos): {val_pos}")
print(f"Clase 0 (negativos): {val_neg}")

El conjunto de entrenamiento presenta un reparto equilibrado entre las dos clases, mientras que el conjunto de validación está algo desbalanceado, con mayor presencia de la clase 0. Aun así, la distribución es adecuada para evaluar el comportamiento del modelo en datos no vistos.

**Distribución de la longitud de los tweet en caracteres:**

In [ ]:
# Longitud de cada tweet (en caracteres)
dataset_train_df["tweet_length"] = dataset_train_df["text"].str.len()

# Estadísticos básicos
print(dataset_train_df["tweet_length"].describe())

# Longitud media
print(f"\nLongitud media: {dataset_train_df['tweet_length'].mean():.2f} caracteres")

# Histograma
plt.figure(figsize=(10,5))
plt.hist(dataset_train_df["tweet_length"], bins=30, edgecolor="black")
plt.title("Distribución de la longitud de los tweets")
plt.xlabel("Número de caracteres")
plt.ylabel("Frecuencia")
plt.show()

**Análisis de contenido de los tweets**


In [ ]:
# Función para limpiar el texto
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)      # Elimina URLs
    text = re.sub(r"@\w+", "", text)         # Elimina menciones (@usuario)
    text = re.sub(r"#", "", text)            # Elimina el símbolo #
    text = re.sub(r"\d+", "", text)          # Elimina números
    text = re.sub(r"[^\w\s]", "", text)      # Elimina signos de puntuación
    return text

# Unimos todos los tweets de cada clase
positive_text = " ".join(
    dataset_train_df[dataset_train_df["label"] == 1]["text"].apply(clean_text)
)

negative_text = " ".join(
    dataset_train_df[dataset_train_df["label"] == 0]["text"].apply(clean_text)
)

# Stopwords
stopwords = set(nltk.corpus.stopwords.words("spanish"))

# Eliminamos términos muy frecuentes del dataset y propios de Twitter, ya que no aportan información discriminativa.
extra_stopwords = {
    "https",
    "http",
    "rt",
    "amp",
    "covid",
    "covid19",
    "coronavirus",
    "día",
    "hoy",
    "va",
    "si",
    "ser",
    "hacer",
    "ahora",
    "dice",
    "puede",
    "solo",
    "tras",
    "asi"
}

stopwords.update(extra_stopwords)

# WordCloud clase positiva
wordcloud_positive = WordCloud(
    width=800,
    height=400,
    background_color="white",
    stopwords=stopwords
).generate(positive_text)

plt.figure(figsize=(12,6))
plt.imshow(wordcloud_positive, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud - Tweets con mención a una profesión (Clase 1)")
plt.show()

# WordCloud clase negativa
wordcloud_negative = WordCloud(
    width=800,
    height=400,
    background_color="white",
    stopwords=stopwords
).generate(negative_text)

plt.figure(figsize=(12,6))
plt.imshow(wordcloud_negative, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud - Tweets sin mención a una profesión (Clase 0)")
plt.show()

## Tokenización

In [ ]:
# Imports
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)


In [ ]:
# Modelo
model_name = "dccuchile/bert-base-spanish-wwm-cased"

He elegido este modelo porque es un modelo BERT preentrenado para español y de tipo encoder, por lo que es adecuado para tareas de clasificación de texto. Además, utiliza la técnica Whole Word Masking, que permite aprender mejores representaciones de las palabras. Al trabajar con tweets en español, este modelo resulta una buena opción para esta tarea.

In [ ]:
# Cargar el tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Función de tokenización
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Tokenizar los tres conjuntos
tokenized_train = dataset["train"].map(tokenize_function, batched=True)
tokenized_val = dataset["validation"].map(tokenize_function, batched=True)
tokenized_test = dataset["test"].map(tokenize_function, batched=True)

## Fine-tuning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

### Configuracion training_args

Configuramos los parámetros de entrenamiento del modelo.


>

In [ ]:

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",

    seed=42,
    full_determinism=True
)

### Métricas de evaluación

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

### Ajuste del modelo

Llevamos a cabo el ajuste del modelo:

In [ ]:
trainer.train()

## Evaluacion

In [ ]:
# Evaluar el modelo sobre el conjunto de validación
results = trainer.evaluate()

print(results)

In [ ]:
predictions = trainer.predict(tokenized_val)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = tokenized_val["label"]

print(classification_report(y_true, y_pred))

Los resultados muestran que el modelo tiene un buen rendimiento en el conjunto de validación. Obtiene una accuracy del 94% y un F1-score de 0.89, por lo que clasifica correctamente la mayoría de los tweets. Además, el classification report muestra que el modelo funciona mejor para la clase 0 que para la clase 1, aunque en esta última consigue un recall alto (0.94), por lo que detecta la mayoría de los casos positivos. En general, las métricas reflejan un buen equilibrio entre precisión y recall y muestran que el modelo es adecuado para esta tarea.

## Genera predicciones

In [ ]:
# Guardar el archivo de predicciones
test_without_labels = tokenized_test.remove_columns("label")

predictions = trainer.predict(test_without_labels)

predicted_labels = np.argmax(predictions.predictions, axis=1)

submission = pd.DataFrame({
    "id": tokenized_test["tweet_id"],
    "label": predicted_labels
})

submission.to_csv(
    "Navarro_Fernandez_Alba_ejercicio1_predicciones.tsv",
    sep="\t",
    index=False
)

submission.head()

from google.colab import files
files.download("Navarro_Fernandez_Alba_ejercicio1_predicciones.tsv")

### Prueba de otros modelos

Se probó el modelo Twitter por estar entrenado con datos de Twitter, lo que podía hacerlo adecuado para esta tarea. Sin embargo, tras el entrenamiento obtuvo un F1-score de 0.759, claramente inferior al conseguido con dccuchile/bert-base-spanish-wwm-cased.

In [ ]:
model_name = "Twitter/twhin-bert-base"

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


tokenized_train = dataset["train"].map(tokenize_function, batched=True)
tokenized_val = dataset["validation"].map(tokenize_function, batched=True)
tokenized_test = dataset["test"].map(tokenize_function, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

In [ ]:

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",

    seed=42,
    full_determinism=True
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:

results = trainer.evaluate()

print(results)

In [ ]:
predictions = trainer.predict(tokenized_val)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = tokenized_val["label"]

print(classification_report(y_true, y_pred))